# Numerai Mission Control

Follow submissions, watch 60-day scores develop, compare models, and track payouts as rounds progress. Public Numerai data powers the dashboard; no API key is needed.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import re

import altair as alt
import mercury as mr
import pandas as pd
import requests
from IPython.display import Markdown, display

API_URL = 'https://api-tournament.numer.ai'
CACHE_DIR = Path.home() / '.cache' / 'numerai-dashboard-advanced'
SNAPSHOT_DIR = Path('numerai-dashboard-advanced-data')
PAYOUT_LOG = CACHE_DIR / 'payout-observations.jsonl'
CACHE_MINUTES = 15
HISTORY_ROUNDS = 180
CACHE_DIR.mkdir(parents=True, exist_ok=True)
alt.data_transformers.disable_max_rows()

PROFILE_QUERY = '''
query($name: String!) { v3UserProfile(modelName: $name) { id username tournament } }
'''
HISTORY_QUERY = '''
query($modelId: String!, $lastNRounds: Int!) {
  v2RoundModelPerformances(modelId: $modelId, tournament: 8, lastNRounds: $lastNRounds) {
    roundNumber roundOpenTime roundResolved submissionId
    tickersAcceptedCount tickersSubmittedCount atRisk payout
    payoutMultipliers { displayName multiplier }
    submissionScores { displayName value percentile day }
    allSubmissionScores { displayName value percentile day date }
  }
}
'''

In [ ]:
def graphql(query, variables):
    response = requests.post(API_URL, json={'query': query, 'variables': variables}, timeout=45)
    response.raise_for_status()
    payload = response.json()
    if payload.get('errors'):
        raise ValueError(payload['errors'])
    return payload['data']

def read_history(path):
    try:
        history = json.loads(path.read_text())
        if (isinstance(history, dict) and isinstance(history.get('rounds'), list)
                and all(isinstance(row, dict) and 'allSubmissionScores' in row
                        and 'roundResolved' in row for row in history['rounds'])):
            return history
    except (OSError, UnicodeError, json.JSONDecodeError):
        pass
    return None

def fetch_model(name):
    profile = graphql(PROFILE_QUERY, {'name': name})['v3UserProfile']
    if not profile or profile['tournament'] != 8:
        raise ValueError(f'No public Numerai Classic model named {name!r}')
    rounds = graphql(HISTORY_QUERY, {'modelId': profile['id'], 'lastNRounds': HISTORY_ROUNDS})['v2RoundModelPerformances']
    return {'name': profile['username'], 'fetched_at': datetime.now(timezone.utc).isoformat(), 'rounds': rounds or []}

def load_model(name, force=False):
    if not re.fullmatch(r'[a-zA-Z0-9_-]+', name):
        raise ValueError(f'Invalid model name: {name!r}')
    cache_file = CACHE_DIR / f'{name}.json'
    cached = read_history(cache_file)
    if cached is not None and not force:
        age = (datetime.now(timezone.utc).timestamp() - cache_file.stat().st_mtime) / 60
        if age < CACHE_MINUTES:
            return cached, 'cache'
    try:
        history = fetch_model(name)
        cache_file.write_text(json.dumps(history))
        return history, 'live GraphQL'
    except (requests.RequestException, ValueError, KeyError):
        if cached is not None:
            return cached, 'stale cache'
        snapshot = read_history(SNAPSHOT_DIR / f'{name}.json')
        if snapshot is not None:
            return snapshot, 'bundled snapshot'
        raise

In [ ]:
model_names = mr.TextInput(
    label='Models (comma-separated)',
    value='v53_lgbm_ender60, integration_test, v53_lgbm_ender20',
)
refresh = mr.Button(label='Refresh from Numerai')

In [ ]:
names = list(dict.fromkeys(name.strip().lower() for name in model_names.value.split(',') if name.strip()))
histories, sources, errors = {}, {}, []
for name in names:
    try:
        histories[name], sources[name] = load_model(name, force=refresh.value)
    except (requests.RequestException, ValueError, KeyError) as error:
        errors.append(f'{name}: {error}')

round_rows, score_rows, daily_rows = [], [], []
for name, history in histories.items():
    for item in history['rounds']:
        latest_day = max((s['day'] or 0 for s in item.get('submissionScores') or []
                          if s['displayName'] in {'corr60', 'mmc60'}), default=0)
        round_rows.append({
            'model': name, 'round': item['roundNumber'], 'resolved': item['roundResolved'],
            'submission_id': item.get('submissionId'),
            'accepted': item.get('tickersAcceptedCount'), 'submitted_rows': item.get('tickersSubmittedCount'),
            'score_day': latest_day,
            'at_risk': float(item['atRisk']) if item.get('atRisk') is not None else None,
            'payout': float(item['payout']) if item.get('payout') is not None else None,
            'multipliers': ', '.join(f"{m['displayName']} x{m['multiplier']:g}"
                                     for m in item.get('payoutMultipliers') or []),
        })
        for score in item.get('submissionScores') or []:
            if score['displayName'] in {'corr60', 'mmc60'} and score['value'] is not None:
                score_rows.append({
                    'model': name, 'round': item['roundNumber'], 'metric': score['displayName'],
                    'score': score['value'], 'day': score['day'], 'resolved': item['roundResolved'],
                    'percentile': score['percentile'],
                })
        for score in item.get('allSubmissionScores') or []:
            if score['displayName'] in {'corr60', 'mmc60'} and score['value'] is not None and score['day']:
                daily_rows.append({
                    'model': name, 'round': item['roundNumber'], 'metric': score['displayName'],
                    'day': score['day'], 'date': score.get('date'), 'score': score['value'],
                })
rounds = pd.DataFrame(round_rows, columns=['model', 'round', 'resolved', 'submission_id', 'accepted',
                                         'submitted_rows', 'score_day', 'at_risk', 'payout', 'multipliers'])
scores = pd.DataFrame(score_rows, columns=['model', 'round', 'metric', 'score', 'day', 'resolved', 'percentile'])
daily = pd.DataFrame(daily_rows, columns=['model', 'round', 'metric', 'day', 'date', 'score'])

In [ ]:
def read_payout_log():
    if not PAYOUT_LOG.exists():
        return []
    rows = []
    for line in PAYOUT_LOG.read_text().splitlines():
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            continue  # An interrupted append must not stop the dashboard.
    return rows

observations = read_payout_log()
latest_observation = {(row['model'], row['round']): row for row in observations}
new_observations = []
for row in rounds.itertuples():
    if sources.get(row.model) != 'live GraphQL' or row.resolved or pd.isna(row.payout) or not row.score_day:
        continue
    previous = latest_observation.get((row.model, row.round))
    if previous and previous['score_day'] == row.score_day and previous['payout'] == row.payout:
        continue
    new_observations.append({
        'observed_at': datetime.now(timezone.utc).isoformat(), 'model': row.model,
        'round': int(row.round), 'score_day': int(row.score_day), 'payout': float(row.payout),
    })
if new_observations:
    with PAYOUT_LOG.open('a') as output:
        for observation in new_observations:
            output.write(json.dumps(observation) + '\n')
    observations.extend(new_observations)

In [ ]:
focus_choices = list(histories) or ['No models loaded']
focus_model = mr.Select(label='Focus model', choices=focus_choices, value=focus_choices[0])

In [ ]:
scored_rounds = (rounds[(rounds['model'] == focus_model.value) & (rounds['score_day'] > 0)]
                 .sort_values('round', ascending=False)['round'].head(25).astype(str).tolist())
round_choices = scored_rounds or ['No scored rounds']
active = rounds[(rounds['model'] == focus_model.value) & (rounds['score_day'] > 0)
                & (rounds['resolved'] == False) & (rounds['round'].astype(str).isin(round_choices))]
default_round = str(int(active.sort_values('score_day', ascending=False).iloc[0]['round'])) if len(active) else round_choices[0]
focus_round = mr.Select(label='Inspect round', choices=round_choices, value=default_round)

In [ ]:
metric = mr.Select(label='Score', choices=['CORR60', 'MMC60'], value='CORR60')

In [ ]:
model_rounds = rounds[rounds['model'] == focus_model.value].sort_values('round', ascending=False)
submitted = model_rounds[model_rounds['submission_id'].notna()]
selected = model_rounds[model_rounds['round'].astype(str) == focus_round.value]
selected_row = selected.iloc[0] if len(selected) else None
payout_value = selected_row['payout'] if selected_row is not None else None
payout_label = 'Final payout (NMR)' if selected_row is not None and selected_row['resolved'] else 'Current payout (NMR, provisional)'
mr.Indicator([
    mr.Indicator(value=str(len(histories)), label='Models'),
    mr.Indicator(value=str(int(submitted.iloc[0]['round'])) if len(submitted) else 'n/a', label='Latest submission round'),
    mr.Indicator(value=str(int(selected_row['score_day'])) if selected_row is not None else 'n/a', label='Selected score day'),
    mr.Indicator(value=f'{payout_value:+.4f}' if pd.notna(payout_value) else 'n/a', label=payout_label),
])

In [ ]:
display(Markdown('## Submissions'))
if model_rounds.empty:
    display(Markdown('No public round history for this model.'))
else:
    recent = model_rounds.head(12).copy()
    recent['state'] = recent['resolved'].map({True: 'Resolved', False: 'Scoring / open'})
    recent['submission'] = recent['submission_id'].map(lambda value: 'Yes' if pd.notna(value) else 'No')
    recent['ID'] = recent['submission_id'].map(lambda value: str(value)[:8] if pd.notna(value) else '-')
    recent['accepted / sent'] = recent.apply(
        lambda row: f"{int(row['accepted']):,} / {int(row['submitted_rows']):,}"
        if pd.notna(row['accepted']) and pd.notna(row['submitted_rows']) else 'n/a', axis=1
    )
    display(recent[['round', 'submission', 'ID', 'accepted / sent', 'score_day', 'state', 'at_risk', 'payout', 'multipliers']]
            .rename(columns={'round': 'Round', 'submission': 'Submitted', 'score_day': 'Score day',
                             'state': 'Round state', 'at_risk': 'At risk (NMR)', 'payout': 'Payout (NMR)',
                             'multipliers': 'Selected multipliers'})
            .style.hide(axis='index').format({'At risk (NMR)': '{:.3f}', 'Payout (NMR)': '{:+.4f}'}, na_rep='-'))

In [ ]:
display(Markdown('## Compare models'))
metric_name = metric.value.lower()
final_scores = scores[(scores['metric'] == metric_name) & (scores['resolved'] == True) & (scores['day'] >= 60)].copy()
round_sets = [set(final_scores.loc[final_scores['model'] == name, 'round']) for name in histories]
common_rounds = set.intersection(*round_sets) if round_sets else set()
comparison = final_scores[final_scores['round'].isin(common_rounds)].copy()
if comparison.empty:
    display(Markdown('No common final 60-day rounds yet for these models.'))
else:
    summary = (comparison.groupby('model', as_index=False)
               .agg(rounds=('round', 'nunique'), mean_score=('score', 'mean'),
                    mean_percentile=('percentile', 'mean'))
               .sort_values('mean_score', ascending=False))
    display(summary.style.hide(axis='index').format({'mean_score': '{:+.4f}',
                                                   'mean_percentile': '{:.1%}'}, na_rep='-'))
    chart = alt.Chart(comparison).mark_line(point=True).encode(
        x=alt.X('round:Q', title='Round', scale=alt.Scale(zero=False)),
        y=alt.Y('score:Q', title=metric.value, scale=alt.Scale(zero=True)),
        color=alt.Color('model:N', title='Model', scale=alt.Scale(scheme='tableau10')),
        tooltip=['model:N', 'round:Q', alt.Tooltip('score:Q', format='.4f'),
                 alt.Tooltip('percentile:Q', format='.1%')],
    ).properties(width='container', height=300)
    display(chart)
display(Markdown("Rounds before 1343 used a 20-day payout target. Their CORR60/MMC60 here are 60-day diagnostics, not payout scores for those rounds."))

In [ ]:
display(Markdown('## How an early score changes'))
progress = daily[(daily['model'] == focus_model.value) & (daily['round'].astype(str) == focus_round.value)
                 & (daily['metric'] == metric_name)].sort_values('day').drop_duplicates('day', keep='last')
if progress.empty:
    display(Markdown('No daily scores yet for this round and metric.'))
else:
    first, last = progress.iloc[0], progress.iloc[-1]
    display(Markdown(f"**{focus_model.value}, round {focus_round.value}:** "
                     f"day {int(first['day'])} {first['score']:+.4f} to "
                     f"day {int(last['day'])} {last['score']:+.4f}."))
    progress_chart = alt.Chart(progress).mark_line(point=True, color='#087e79').encode(
        x=alt.X('day:Q', title='Score day', scale=alt.Scale(zero=False)),
        y=alt.Y('score:Q', title=metric.value, scale=alt.Scale(zero=True)),
        tooltip=['day:Q', 'date:N', alt.Tooltip('score:Q', format='.4f')],
    ).properties(width='container', height=280)
    display(progress_chart)

In [ ]:
display(Markdown('## Payouts and burns'))
payouts = model_rounds[model_rounds['payout'].notna()].head(45).copy()
if payouts.empty:
    display(Markdown('No API payout values for this model yet.'))
else:
    payouts['state'] = payouts['resolved'].map({True: 'Resolved', False: 'Provisional'})
    payout_chart = alt.Chart(payouts).mark_bar().encode(
        x=alt.X('round:O', title='Round', sort='ascending'),
        y=alt.Y('payout:Q', title='API payout (NMR)'),
        color=alt.Color('state:N', scale=alt.Scale(
            domain=['Resolved', 'Provisional'], range=['#087e79', '#e59b3d'])),
        tooltip=['round:Q', 'state:N', alt.Tooltip('payout:Q', format='+.4f'),
                 alt.Tooltip('at_risk:Q', title='At risk (NMR)', format='.3f'), 'multipliers:N'],
    ).properties(width='container', height=300)
    display(payout_chart)
display(Markdown('Unresolved payout values are **provisional**, not settled NMR. Selected multipliers vary by round; this app reports the API value rather than recalculating a payout.'))

In [ ]:
display(Markdown('## How the payout estimate changes'))
history = pd.DataFrame(observations)
if not history.empty:
    history = history[(history['model'] == focus_model.value)
                      & (history['round'].astype(str) == focus_round.value)].copy()
if len(history) < 2:
    display(Markdown('This view starts collecting public payout observations when you refresh the app. Return after another scoring update to see the change.'))
else:
    history['observed_at'] = pd.to_datetime(history['observed_at'], utc=True)
    estimate_chart = alt.Chart(history).mark_line(point=True, color='#b65b31').encode(
        x=alt.X('observed_at:T', title='Observed at (UTC)'),
        y=alt.Y('payout:Q', title='Provisional API payout (NMR)'),
        tooltip=[alt.Tooltip('observed_at:T', title='Observed at'), 'score_day:Q',
                 alt.Tooltip('payout:Q', format='+.4f')],
    ).properties(width='container', height=260)
    display(estimate_chart)
if errors:
    display(Markdown('**Unavailable models:** ' + '; '.join(errors)))
display(Markdown('**Data sources:** ' + ', '.join(f'{name}: {source}' for name, source in sources.items())))